# CU28 mixed_context - External Context EDA

Notebook narrativo de auditoria para el scope `mixed_context`.


## Objetivo

Analizar las series procesadas de contexto externo para entender que variables proxy alimentan el pipeline oficial y cuales son sus limitaciones.


## Alcance

Este analisis describe la ruta oficial reproducible `mixed_context`. Las senales externas se tratan como contexto/proxy. Las variables internas de planta siguen siendo sinteticas salvo carga posterior de cliente.


## Inputs

            - `data/processed/external/context/external_long.csv`
- `data/processed/external/context/context_weekly_for_simulation.csv`
- `data/processed/external/context/context_proxy_limitations.json`


## Outputs esperados

            - `reports/tables/eda/external_context_summary__mixed_context.csv`
- `reports/tables/eda/external_context_weekly_snapshot__mixed_context.csv`
- `reports/tables/eda/external_context_limitations__mixed_context.csv`
- `reports/figures/eda/external_context_demand_index__mixed_context.png`
- `reports/figures/eda/external_context_supply_index__mixed_context.png`
- `reports/figures/eda/external_context_gap__mixed_context.png`
- `reports/figures/eda/external_context_price_index__mixed_context.png`
- `reports/figures/eda/external_context_correlation__mixed_context.png`
- `reports/figures/eda/external_context_coverage__mixed_context.png`


## Limitaciones

Este notebook documenta evidencia reproducible del pipeline oficial, pero no sustituye la revision de codigo, la auditoria de datos de origen ni una certificacion operacional de planta.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.reproducibility.notebook_support import (
    detect_temporal_columns,
    ensure_eda_dirs,
    execution_metadata,
    first_valid_temporal_range,
    load_source_manifests,
    load_tabular_file,
    parse_markdown_table,
    print_frame,
    print_series,
    project_root,
    read_json,
    relative_to_root,
    save_figure,
    save_table,
    sha256_file,
)


In [ ]:
NOTEBOOK_NAME = "02_external_context_eda.ipynb"
PROJECT_ROOT = project_root()
SCOPE = globals().get("scope", "mixed_context")
REPORT_DIRS = ensure_eda_dirs()
META = execution_metadata(SCOPE)
FIGURES = []
TABLES = []
print(json.dumps(META, indent=2))


## Carga de datos

Las siguientes celdas cargan los artefactos de entrada y muestran verificaciones intermedias antes de producir tablas y graficas.


In [ ]:
external_long_path = PROJECT_ROOT / "data/processed/external/context/external_long.csv"
context_weekly_path = PROJECT_ROOT / "data/processed/external/context/context_weekly_for_simulation.csv"
limitations_path = PROJECT_ROOT / "data/processed/external/context/context_proxy_limitations.json"
external_long = pd.read_csv(external_long_path)
external_long["date"] = pd.to_datetime(external_long["date"], errors="coerce")
print(external_long.shape)
print(external_long.columns.tolist())
print_frame("external_long preview", external_long.head(10))


In [ ]:
context_weekly = pd.read_csv(context_weekly_path)
context_weekly["date"] = pd.to_datetime(context_weekly["date"], errors="coerce")
context_weekly["demand_supply_gap"] = context_weekly["demand_index"] - context_weekly["supply_index"]
limitations_payload = read_json(limitations_path)
limitations_df = pd.DataFrame({"limitation": limitations_payload.get("limitations", [])})
print(context_weekly.shape)
print(context_weekly.columns.tolist())
print_frame("context_weekly preview", context_weekly.head(10))


## Inspeccion inicial

Primero se comprueba la estructura de ambos datasets procesados y el rango temporal disponible para la simulacion semanal.


In [ ]:
external_summary_shape = pd.DataFrame(
    [{"dataset": "external_long", "rows": len(external_long), "columns": len(external_long.columns)}]
)
print_frame("Shape of external_long", external_summary_shape)
display(external_summary_shape)


In [ ]:
weekly_summary_shape = pd.DataFrame(
    [{"dataset": "context_weekly", "rows": len(context_weekly), "columns": len(context_weekly.columns)}]
)
print_frame("Shape of context_weekly", weekly_summary_shape)
display(weekly_summary_shape)


In [ ]:
temporal_range = pd.DataFrame(
    [
        {
            "dataset": "external_long",
            "date_min": str(external_long["date"].min().date()),
            "date_max": str(external_long["date"].max().date()),
        },
        {
            "dataset": "context_weekly",
            "date_min": str(context_weekly["date"].min().date()),
            "date_max": str(context_weekly["date"].max().date()),
        },
    ]
)
variable_table = pd.DataFrame({"variable": context_weekly.columns})
print_frame("Temporal range", temporal_range)
print_frame("Variables in context_weekly", variable_table, rows=20)


In [ ]:
missing_summary = context_weekly.isna().mean().reset_index()
missing_summary.columns = ["variable", "missing_pct"]
missing_summary["missing_pct"] = missing_summary["missing_pct"].round(4)
print_frame("Missing values in context_weekly", missing_summary, rows=20)
display(missing_summary)


## Tablas intermedias

Se resume `external_long` por fuente, dataset y subserie para ver observaciones, cobertura y huecos antes de analizar las series agregadas semanales.


In [ ]:
external_context_summary = (
    external_long.groupby(["source", "dataset", "subseries"], dropna=False)
    .agg(
        variable=("unit", "first"),
        min_date=("date", "min"),
        max_date=("date", "max"),
        observations=("value", "size"),
        missing_rate=("value", lambda s: float(pd.to_numeric(s, errors="coerce").isna().mean())),
    )
    .reset_index()
)
print_frame("Summary by source, dataset and subseries", external_context_summary, rows=20)
display(external_context_summary.head(20))


In [ ]:
weekly_snapshot = context_weekly[["date", "demand_index", "supply_index", "demand_supply_gap", "purchase_price_index"]].head(12).copy()
print_frame("Weekly snapshot", weekly_snapshot, rows=12)
display(weekly_snapshot)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(context_weekly["date"], context_weekly["demand_index"], color="#355c7d")
ax.set_title("Demand index over time")
ax.set_ylabel("index")
FIGURES.append(save_figure(fig, "external_context_demand_index__mixed_context.png"))
plt.close(fig)


### Interpretacion de la figura

`demand_index` captura presion de demanda sectorial agregada. Es una senal proxy de contexto y no una observacion directa de pedidos internos de planta.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(context_weekly["date"], context_weekly["supply_index"], color="#6c5b7b")
ax.set_title("Supply index over time")
ax.set_ylabel("index")
FIGURES.append(save_figure(fig, "external_context_supply_index__mixed_context.png"))
plt.close(fig)


### Interpretacion de la figura

`supply_index` resume un proxy de oferta sectorial. Su lectura debe hacerse como contexto macro, no como disponibilidad confirmada para una planta concreta.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(context_weekly["date"], context_weekly["demand_supply_gap"], color="#c06c84")
ax.axhline(0.0, linestyle="--", color="black", linewidth=1)
ax.set_title("Demand supply gap over time")
ax.set_ylabel("gap")
FIGURES.append(save_figure(fig, "external_context_gap__mixed_context.png"))
plt.close(fig)


### Interpretacion de la figura

La brecha `demand_supply_gap` representa tension relativa entre demanda y oferta. Se usa como contexto de riesgo, no como decision de compra final.


In [ ]:
purchase_price_constant = context_weekly["purchase_price_index"].nunique() == 1
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(context_weekly["date"], context_weekly["purchase_price_index"], color="#f67280")
ax.set_title("Purchase price index")
ax.set_ylabel("index")
FIGURES.append(save_figure(fig, "external_context_price_index__mixed_context.png"))
plt.close(fig)
print(f"purchase_price_index uses fallback constant: {purchase_price_constant}")


In [ ]:
corr = context_weekly[["demand_index", "supply_index", "purchase_price_index", "demand_supply_gap"]].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Correlation between external signals")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
FIGURES.append(save_figure(fig, "external_context_correlation__mixed_context.png"))
plt.close(fig)
print(corr.to_string())


In [ ]:
coverage_df = context_weekly.notna().mean().reset_index()
coverage_df.columns = ["variable", "coverage_rate"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(coverage_df["variable"], coverage_df["coverage_rate"], color="#2a9d8f")
ax.set_title("Weekly coverage by variable")
ax.set_ylabel("coverage_rate")
ax.tick_params(axis="x", rotation=30)
FIGURES.append(save_figure(fig, "external_context_coverage__mixed_context.png"))
plt.close(fig)
print_frame("Coverage table", coverage_df)


## Limitaciones proxy

La siguiente tabla se deriva del JSON de limitaciones y debe leerse junto con la evidencia de que `purchase_price_index` puede operar como valor constante de respaldo.


In [ ]:
print_frame("Proxy limitations", limitations_df, rows=20)
display(limitations_df)


In [ ]:
TABLES.append(save_table(external_context_summary, "external_context_summary__mixed_context.csv"))
TABLES.append(save_table(weekly_snapshot, "external_context_weekly_snapshot__mixed_context.csv"))
TABLES.append(save_table(limitations_df, "external_context_limitations__mixed_context.csv"))
RESULT = {
    "notebook": NOTEBOOK_NAME,
    "tables": TABLES,
    "figures": FIGURES,
    "findings": [
        "External processed signals remain contextual proxies and do not replace internal plant telemetry.",
        "The weekly dataset is compact and reproducible, with demand, supply and gap signals aligned on a common calendar.",
        "purchase_price_index must be interpreted as fallback evidence when it remains constant.",
    ],
    "limitations": [
        "Proxy limitations are structural and already documented in the dedicated JSON contract.",
    ],
}
print(json.dumps(RESULT, indent=2))


## Concluson final

`external_long` documenta procedencia y cobertura de las series fuente, mientras que `context_weekly_for_simulation.csv` concentra el contexto semanal que alimenta la reconstruccion `mixed_context`.
